# 模型导出教程 (Model Export Tutorial)

本教程详细介绍深度学习模型的导出技术，包括：

1. **ONNX 导出**: 跨框架通用格式
2. **TorchScript 导出**: PyTorch 原生格式
3. **模型分析**: 参数统计和性能分析
4. **导出验证**: 确保导出正确性

---

## 为什么需要模型导出？

| 场景 | 训练环境 | 部署环境 |
|:-----|:---------|:---------|
| 框架 | PyTorch/TensorFlow | ONNX Runtime/TensorRT |
| 语言 | Python | C++/Java/Go |
| 设备 | GPU 服务器 | 边缘设备/手机 |

In [ ]:
import sys
sys.path.insert(0, '../src')

import torch
import torch.nn as nn
import torch.nn.functional as F
import tempfile
import os

torch.manual_seed(42)
print(f"PyTorch 版本: {torch.__version__}")

## 1. 定义测试模型

In [ ]:
class ImageClassifier(nn.Module):
    """图像分类模型"""
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.pool = nn.MaxPool2d(2)
        self.fc1 = nn.Linear(64 * 8 * 8, 256)
        self.fc2 = nn.Linear(256, num_classes)
        self.dropout = nn.Dropout(0.5)
    
    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = x.view(x.size(0), -1)
        x = self.dropout(F.relu(self.fc1(x)))
        return self.fc2(x)

model = ImageClassifier()
model.eval()

# 示例输入
dummy_input = torch.randn(1, 3, 32, 32)

# 测试前向传播
with torch.no_grad():
    output = model(dummy_input)
print(f"模型输出形状: {output.shape}")

## 2. 模型分析

在导出前，先分析模型的结构和性能。

In [ ]:
from export import ModelAnalyzer

# 参数统计
param_stats = ModelAnalyzer.count_parameters(model)

print("模型参数统计:")
print(f"  总参数量: {param_stats['total']:,}")
print(f"  可训练参数: {param_stats['trainable']:,}")
print(f"\n各层参数量:")
for name, count in param_stats['layers'].items():
    print(f"  {name}: {count:,}")

In [ ]:
# 模型大小估算
size_fp32 = ModelAnalyzer.estimate_model_size(model, torch.float32)
size_fp16 = ModelAnalyzer.estimate_model_size(model, torch.float16)
size_int8 = ModelAnalyzer.estimate_model_size(model, torch.int8)

print("模型大小估算:")
print(f"  FP32: {size_fp32['total_mb']:.2f} MB")
print(f"  FP16: {size_fp16['total_mb']:.2f} MB")
print(f"  INT8: {size_int8['total_mb']:.2f} MB")

In [ ]:
# 推理性能分析
perf_stats = ModelAnalyzer.profile_inference(
    model,
    input_shape=(1, 3, 32, 32),
    num_runs=50,
    warmup_runs=10,
    device="cpu"
)

print("推理性能 (CPU):")
print(f"  平均延迟: {perf_stats['mean_ms']:.2f} ms")
print(f"  最小延迟: {perf_stats['min_ms']:.2f} ms")
print(f"  最大延迟: {perf_stats['max_ms']:.2f} ms")
print(f"  吞吐量: {perf_stats['throughput']:.1f} samples/sec")

## 3. TorchScript 导出

TorchScript 是 PyTorch 的原生序列化格式，支持两种方式：

- **Tracing**: 记录执行路径
- **Scripting**: 分析 Python 代码

In [ ]:
from export import TorchScriptExporter, export_to_torchscript

# 方法 1: Tracing (追踪)
print("=== TorchScript Tracing ===")
traced_model = torch.jit.trace(model, dummy_input)

# 验证输出
with torch.no_grad():
    original_out = model(dummy_input)
    traced_out = traced_model(dummy_input)

print(f"输出一致: {torch.allclose(original_out, traced_out, atol=1e-5)}")
print(f"最大差异: {(original_out - traced_out).abs().max():.6f}")

In [ ]:
# 方法 2: Scripting (脚本化)
print("=== TorchScript Scripting ===")
scripted_model = torch.jit.script(model)

# 验证输出
with torch.no_grad():
    scripted_out = scripted_model(dummy_input)

print(f"输出一致: {torch.allclose(original_out, scripted_out, atol=1e-5)}")
print(f"最大差异: {(original_out - scripted_out).abs().max():.6f}")

In [ ]:
# 查看生成的代码
print("TorchScript 生成的代码:")
print(scripted_model.code)

In [ ]:
# 保存和加载
with tempfile.TemporaryDirectory() as tmpdir:
    # 保存
    save_path = os.path.join(tmpdir, "model.pt")
    traced_model.save(save_path)
    
    # 检查文件大小
    file_size = os.path.getsize(save_path) / (1024 * 1024)
    print(f"保存的模型大小: {file_size:.2f} MB")
    
    # 注意: 由于路径问题，这里不演示加载
    # 在实际使用中可以用 torch.jit.load(save_path) 加载

### Tracing vs Scripting

| 特性 | Tracing | Scripting |
|:-----|:--------|:----------|
| 控制流 | 不支持动态 | 支持 |
| 实现 | 记录执行 | 分析代码 |
| 适用 | 静态模型 | 动态模型 |

In [ ]:
# 带控制流的模型示例
class DynamicModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(10, 10)
    
    def forward(self, x, use_relu: bool = True):
        x = self.fc(x)
        if use_relu:  # 动态控制流
            x = F.relu(x)
        return x

dynamic_model = DynamicModel()

# Scripting 可以处理控制流
scripted_dynamic = torch.jit.script(dynamic_model)

x = torch.randn(1, 10)
print("Scripting 支持动态控制流:")
print(f"  use_relu=True: {scripted_dynamic(x, True).shape}")
print(f"  use_relu=False: {scripted_dynamic(x, False).shape}")

## 4. ONNX 导出

ONNX (Open Neural Network Exchange) 是跨框架的通用格式。

In [ ]:
# 检查 ONNX 是否可用
try:
    import onnx
    ONNX_AVAILABLE = True
    print(f"ONNX 版本: {onnx.__version__}")
except ImportError:
    ONNX_AVAILABLE = False
    print("ONNX 未安装，跳过 ONNX 相关示例")
    print("安装命令: pip install onnx onnxruntime")

In [ ]:
if ONNX_AVAILABLE:
    with tempfile.TemporaryDirectory() as tmpdir:
        onnx_path = os.path.join(tmpdir, "model.onnx")
        
        # 导出
        torch.onnx.export(
            model,
            dummy_input,
            onnx_path,
            input_names=['input'],
            output_names=['output'],
            dynamic_axes={
                'input': {0: 'batch_size'},
                'output': {0: 'batch_size'}
            },
            opset_version=14
        )
        
        # 验证
        onnx_model = onnx.load(onnx_path)
        onnx.checker.check_model(onnx_model)
        
        print("ONNX 导出成功!")
        print(f"文件大小: {os.path.getsize(onnx_path) / (1024*1024):.2f} MB")
else:
    print("跳过 ONNX 导出示例")

In [ ]:
# 使用 ONNX Runtime 推理
if ONNX_AVAILABLE:
    try:
        import onnxruntime as ort
        
        with tempfile.TemporaryDirectory() as tmpdir:
            onnx_path = os.path.join(tmpdir, "model.onnx")
            torch.onnx.export(model, dummy_input, onnx_path, opset_version=14)
            
            # 创建推理会话
            session = ort.InferenceSession(onnx_path)
            
            # 推理
            input_name = session.get_inputs()[0].name
            ort_output = session.run(None, {input_name: dummy_input.numpy()})[0]
            
            # 比较
            with torch.no_grad():
                torch_output = model(dummy_input).numpy()
            
            diff = abs(torch_output - ort_output).max()
            print(f"PyTorch vs ONNX Runtime 最大差异: {diff:.6f}")
            print(f"输出一致: {diff < 1e-5}")
    except ImportError:
        print("ONNX Runtime 未安装")

## 5. 导出最佳实践

In [ ]:
# 导出前的检查清单
def pre_export_checklist(model, dummy_input):
    """导出前检查"""
    print("导出前检查清单:")
    print("=" * 40)
    
    # 1. 模型模式
    is_eval = not model.training
    print(f"[{'✓' if is_eval else '✗'}] 模型处于 eval 模式")
    
    # 2. 前向传播测试
    try:
        with torch.no_grad():
            _ = model(dummy_input)
        print("[✓] 前向传播正常")
    except Exception as e:
        print(f"[✗] 前向传播失败: {e}")
    
    # 3. 输入形状
    print(f"[i] 输入形状: {dummy_input.shape}")
    
    # 4. 参数统计
    total_params = sum(p.numel() for p in model.parameters())
    print(f"[i] 总参数量: {total_params:,}")
    
    # 5. 检查 NaN/Inf
    has_nan = any(torch.isnan(p).any() for p in model.parameters())
    has_inf = any(torch.isinf(p).any() for p in model.parameters())
    print(f"[{'✗' if has_nan else '✓'}] 无 NaN 参数")
    print(f"[{'✗' if has_inf else '✓'}] 无 Inf 参数")
    
    print("=" * 40)
    return is_eval and not has_nan and not has_inf

# 运行检查
model.eval()
ready = pre_export_checklist(model, dummy_input)
print(f"\n准备导出: {'是' if ready else '否'}")

In [ ]:
# 模型输出比较工具
from export import compare_model_outputs

# 比较原始模型和 TorchScript 模型
test_inputs = [torch.randn(1, 3, 32, 32) for _ in range(10)]

results = compare_model_outputs(
    model, traced_model, test_inputs,
    atol=1e-5, rtol=1e-5
)

print("模型输出比较结果:")
print(f"  测试样本数: {results['num_tests']}")
print(f"  全部匹配: {results['all_match']}")
print(f"  最大差异: {results['max_diff']:.6f}")
print(f"  平均差异: {results['mean_diff']:.6f}")

## 6. 导出格式对比

| 格式 | 优点 | 缺点 | 适用场景 |
|:-----|:-----|:-----|:---------|
| TorchScript | PyTorch 原生 | 仅 PyTorch | PyTorch 部署 |
| ONNX | 跨框架 | 部分算子不支持 | 通用部署 |
| TensorRT | 高性能 | 仅 NVIDIA | GPU 推理 |

In [ ]:
# 性能对比
import time

def benchmark(model_fn, input_tensor, num_runs=100, name="Model"):
    """基准测试"""
    # 预热
    for _ in range(10):
        _ = model_fn(input_tensor)
    
    # 计时
    times = []
    for _ in range(num_runs):
        start = time.perf_counter()
        _ = model_fn(input_tensor)
        times.append((time.perf_counter() - start) * 1000)
    
    return {
        'name': name,
        'mean': sum(times) / len(times),
        'min': min(times),
        'max': max(times)
    }

# 测试
test_input = torch.randn(1, 3, 32, 32)

with torch.no_grad():
    results = [
        benchmark(model, test_input, name="PyTorch (eager)"),
        benchmark(traced_model, test_input, name="TorchScript (trace)"),
        benchmark(scripted_model, test_input, name="TorchScript (script)")
    ]

print("推理性能对比 (ms):")
print(f"{'模型':<25} {'平均':<10} {'最小':<10} {'最大':<10}")
print("-" * 55)
for r in results:
    print(f"{r['name']:<25} {r['mean']:<10.3f} {r['min']:<10.3f} {r['max']:<10.3f}")

## 总结

本教程介绍了模型导出的核心概念：

1. **模型分析**: 导出前了解模型结构和性能
2. **TorchScript**: PyTorch 原生格式，支持 trace 和 script
3. **ONNX**: 跨框架通用格式
4. **验证**: 确保导出模型输出正确

### 选择建议

- **PyTorch 生态**: TorchScript
- **跨框架部署**: ONNX
- **NVIDIA GPU**: TensorRT
- **移动端**: ONNX + NNAPI/CoreML